In [1]:
%pip install azure-search-documents==11.6.0b9 azure-identity python-dotenv pandas jinja2 --quiet

Note: you may need to restart the kernel to use updated packages.


In [ ]:
AZURE_SEARCH_SERVICE_ENDPOINT="https://testingitplatzi.search.windows.net"
AZURE_SEARCH_INDEX="rag-testing2"
AZURE_SEARCH_ADMIN_KEY="<coloca aquí tu clave de Azure Search>"

In [ ]:
import os
from azure.core.credentials import AzureKeyCredential
from azure.identity import DefaultAzureCredential
from dotenv import load_dotenv

load_dotenv(override=True) # toma las variables de entorno del archivo .env

endpoint = AZURE_SEARCH_SERVICE_ENDPOINT
index_name = AZURE_SEARCH_INDEX
credential = AzureKeyCredential(AZURE_SEARCH_ADMIN_KEY)

In [3]:
from azure.search.documents import SearchClient
import pandas as pd

search_client = SearchClient(endpoint, index_name, credential)

def display_results(results):
    df = pd.json_normalize(list(results)).dropna(axis=1, how='all')
    df["chunk"] = df["chunk"].apply(lambda c: c[:300] + '...' if len(c) > 300 else c)
    first_cols = ['title', 'chunk', '@search.score']
    df = df[first_cols + [col for col in df.columns if col not in first_cols]]

    df = df.style.set_properties(**{
        'max-width': '500px',
        'text-align': 'left',
        'white-space': 'normal',
        'word-wrap': 'break-word'
    }).hide(axis="index")


    return df

In [4]:
results = search_client.search(search_text="What is Contoso", top=5, select=["title", "chunk"])

display_results(results)

title,chunk,@search.score
Northwind_Health_Plus_Benefits_Details.pdf,"This means that you must obtain approval from Northwind Health Plus prior to receiving the service. If pre-authorization or pre-certification is not obtained, you may be responsible for the full cost of the services. It is important to understand that the Allowed Amount does not include any...",4.179370
Northwind_Standard_Benefits_Details.pdf,"your coverage, please contact our customer service department at Contoso. Neurodevelopmental Therapy (Habilitation) Neurodevelopmental Therapy (Habilitation) At Contoso, we want to ensure that our employees and their dependents receive the best possible care and services. That’s why we’ve ...",4.075625
Northwind_Health_Plus_Benefits_Details.pdf,"the tips outlined above, you can help ensure that your request for services or treatments is approved in a timely manner and that you are receiving the most appropriate care. The Group And You OTHER INFORMATION ABOUT THIS PLAN The Group and You The Northwind Health Plus plan is a gro...",3.811434
Northwind_Standard_Benefits_Details.pdf,"providers that are not available from participating providers. Additionally, in some cases, the health plan may cover non-participating providers’ charges if there are no participating providers in your area. Tips In order to avoid costly balance billing amounts, it is important to make sure...",3.709958
Northwind_Standard_Benefits_Details.pdf,"At Contoso, we understand that medical costs can be intimidating and confusing, which is why we’ve partnered with Northwind Health to offer our employees the Northwind Standard plan. This plan provides a balance billing protection, meaning that you are protected from unexpected costs when visi...",3.431380


In [5]:
from azure.search.documents.models import VectorizableTextQuery

results = search_client.search(vector_queries=[VectorizableTextQuery(text="What is Contoso", k_nearest_neighbors=50, fields="text_vector")], top=5, select=["title", "chunk"])

display_results(results)

title,chunk,@search.score
MSFT_cloud_architecture_contoso.pdf,How a fictional but representative global organization has implemented the Microsoft Cloud Contoso in the Microsoft Cloud This topic is 1 of 7 in a series The Contoso Corporation Contoso s worldwide organization Elements of Contoso s implementation of the Microsoft cloud Networking Net...,0.852663
MSFT_cloud_architecture_contoso.pdf,mailto:cloudadopt@microsoft.com mailto:cloudadopt@microsoft.com https://azure.microsoft.com/services/expressroute/ https://azure.microsoft.com/services/expressroute/ https://azure.microsoft.com/services/expressroute/ https://azure.microsoft.com/services/expressroute/ https://aka.ms/o365protect_devic...,0.846799
MSFT_cloud_architecture_contoso.pdf,a fictional but representative global organization has implemented the Microsoft Cloud Contoso s app infrastructure Contoso has the following networking infrastructure. On-premises network WAN links connect the Paris headquarters to regional offices and regional offices to satellite office...,0.844584
MSFT_cloud_architecture_contoso.pdf,have identified the following elements when planning for the adoption of Microsoft s cloud offerings. Microsoft Cloud Identity for Enterprise Architects Microsoft Cloud Identity for Enterprise Architects Microsoft Cloud Networking for Enterprise Architects Microsoft Cloud Networking for...,0.838508
MSFT_cloud_architecture_contoso.pdf,https://technet.microsoft.com/library/mt775341.aspx https://technet.microsoft.com/library/mt775341.aspx How a fictional but representative global organization has implemented the Microsoft Cloud Contoso in the Microsoft Cloud Contoso s IT infrastructure and needs Mapping Contoso s busin...,0.837523


In [6]:
results = search_client.search(
    search_text="What is Contoso",
    vector_queries=[VectorizableTextQuery(text="What is Contoso", k_nearest_neighbors=50, fields="text_vector")],
    top=5,
    select=["title", "chunk"]
)

display_results(results)

title,chunk,@search.score
MSFT_cloud_architecture_contoso.pdf,a fictional but representative global organization has implemented the Microsoft Cloud Contoso s app infrastructure Contoso has the following networking infrastructure. On-premises network WAN links connect the Paris headquarters to regional offices and regional offices to satellite office...,0.026546
MSFT_cloud_architecture_contoso.pdf,"and tenants for Microsoft s cloud offeringsSubscriptions, licenses, accounts, and tenants for Microsoft s cloud offerings Tenants: This topic is 5 of 7 in a series Sales.Production Admin.Production IT.Development IT.Testing IT.Production Sales.Production IT.Production Sales.Production IT....",0.026334
MSFT_cloud_architecture_contoso.pdf,How a fictional but representative global organization has implemented the Microsoft Cloud Contoso in the Microsoft Cloud This topic is 1 of 7 in a series The Contoso Corporation Contoso s worldwide organization Elements of Contoso s implementation of the Microsoft cloud Networking Net...,0.026282
MSFT_cloud_architecture_contoso.pdf,"the Paris headquarters with a high-bandwidth WAN link. Each regional hub has an average of 2,000 workers. Satellite offices contain 80% sales and support staff and provide a physical and on-site presence for Contoso customers in key cities or sub- regions. Each satellite office is ...",0.026254
MSFT_cloud_architecture_contoso.pdf,https://technet.microsoft.com/library/mt775341.aspx https://technet.microsoft.com/library/mt775341.aspx How a fictional but representative global organization has implemented the Microsoft Cloud Contoso in the Microsoft Cloud Contoso s IT infrastructure and needs Mapping Contoso s busin...,0.025726
